#       LightGBM with Quadratic Weighted Kappa (QWK) Optimization

This notebook builds a robust machine learning pipeline for a custom Hackathon dataset.
Because the evaluation metric is Quadratic Weighted Kappa (QWK), we frame this as a regression problem. Instead of classifying directly, we output continuous predictions and use a custom optimizer (Nelder-Mead) to find the perfect thresholds that separate the classes (0, 1, 2, 3), thereby maximizing the QWK score.

Key Highlights:
* Custom threshold optimization for QWK.
* Anomaly extraction (handling string anomalies inside numeric data).
* Dynamic type inference to automatically separate numeric/categorical features.
* Excel-corrupted ID fixing for successful Kaggle submission.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize
from functools import partial
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

## Step 1: Load Datasets
We load the train and test sets from the Kaggle directories.
Crucially, we force column `A` (the ID column) to be read as a string. This prevents pandas from auto-formatting scientific-looking IDs (e.g., `6.42E+11`) into floats, which would break the submission. We also drop any rows in the training set where the target is missing.

In [ ]:
print("Loading datasets...")
train_path = '/kaggle/input/datasets/immihir/hackathon/train.csv'
test_path = '/kaggle/input/datasets/immihir/hackathon/test.csv'

# Force column 'A' (the IDs) to be read as strings to prevent immediate truncation
train = pd.read_csv(train_path, dtype={'A': str})
test = pd.read_csv(test_path, dtype={'A': str})

print(f"Original Train shape: {train.shape}")
print(f"Original Test shape: {test.shape}")

# Clean the target column by dropping rows with missing targets
train['target'] = pd.to_numeric(train['target'], errors='coerce')
train = train.dropna(subset=['target']).reset_index(drop=True)
train['target'] = train['target'].astype(int)

print(f"Cleaned Train shape: {train.shape}")

## Step 2: Custom QWK Optimizer
Since QWK measures agreement between ordinal classes, rounding regression outputs strictly at `0.5, 1.5, 2.5` is often suboptimal. This custom `OptimizedRounder` class uses the Nelder-Mead optimization algorithm to find the exact decimal thresholds that maximize the QWK metric based on out-of-fold predictions.

In [ ]:
class OptimizedRounder:
    """
    Optimizes regression predictions to discrete classes (0,1,2,3)
    to maximize the Quadratic Weighted Kappa score.
    """
    def __init__(self):
        self.coef_ = 0

    def _kappa_loss(self, coef, X, y):
        X_p = np.digitize(X, coef)
        ll = cohen_kappa_score(y, X_p, weights='quadratic')
        return -ll

    def fit(self, X, y):
        loss_partial = partial(self._kappa_loss, X=X, y=y)
        initial_coef = [0.5, 1.5, 2.5]
        self.coef_ = minimize(loss_partial, initial_coef, method='nelder-mead')

    def predict(self, X, coef):
        return np.digitize(X, coef)

## Step 3: Feature Engineering & Anomaly Handling
The dataset contains strange textual anomalies (like `'17cm'`, `'chotu'`, etc.) scattered across features. Instead of just replacing them with NaNs, we treat their presence as potential signal. We create count features for these anomalies and a count for missing values.

In [ ]:
print("Engineering features...")

# List of known anomalies found in the dataset
cal_codes = ['17cm', '500years', 'chotu', 'rinki']

for df in [train, test]:
    for code in cal_codes:
        # Count occurrences of these codes per row
        df[f'count_{code}'] = (df == code).sum(axis=1)
    # Count missing values per row
    df['missing_count'] = df.isna().sum(axis=1)

# Define feature list (excluding ID and target)
features = [c for c in train.columns if c not in ['A', 'target']]

## Step 4: Dynamic Type Inference & Cleaning
Because of the text anomalies, pandas might have read numeric columns as `object`. Here we dynamically scan every column. If a column is mostly numeric (once anomalies are ignored), we cast it back to numeric. Otherwise, we treat it as categorical and apply Label Encoding.

In [ ]:
print("Parsing column types dynamically...")
num_cols = []
cat_cols = []

for col in features:
    # Always treat our engineered counts as numeric
    if col.startswith('count_') or col == 'missing_count':
        num_cols.append(col)
        continue

    # Filter out anomalies and missing values to check the true data type
    valid_vals = train[col][~train[col].isin(cal_codes) & train[col].notna()]
    num_converted = pd.to_numeric(valid_vals, errors='coerce')

    # If more than 50% of the valid values can be cast to float, it's numeric
    if len(valid_vals) > 0 and num_converted.notna().mean() > 0.5:
        num_cols.append(col)
    else:
        cat_cols.append(col)

print(f"Total features: {len(features)} ({len(num_cols)} Numeric, {len(cat_cols)} Categorical)")

# Apply conversions
for df in [train, test]:
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in cat_cols:
        df[col] = df[col].fillna('missing').astype(str)

# Label Encode categorical columns
for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined data to ensure all classes are captured
    le.fit(list(train[col]) + list(test[col]))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

## Step 5: LightGBM Model Training (Stratified K-Fold)
We use a 5-fold Stratified K-Fold cross-validation strategy. LightGBM is trained in regression mode (`objective='regression'`). The out-of-fold (OOF) predictions are stored for threshold optimization, and test predictions are averaged across all 5 folds.

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 7,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'subsample': 0.8,
    'seed': 42,
    'verbose': -1
}

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(train, train['target'])):
    print(f"\n--- Training Fold {fold + 1} ---")

    X_train, y_train = train.loc[train_idx, features], train.loc[train_idx, 'target']
    X_val, y_val = train.loc[val_idx, features], train.loc[val_idx, 'target']

    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
    val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, free_raw_data=False)

    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=2000,
        valid_sets=[train_data, val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(test[features], num_iteration=model.best_iteration) / kf.n_splits

## Step 6: Threshold Optimization & Submission
We feed our continuous out-of-fold predictions into the `OptimizedRounder`. Once the optimal thresholds are found, we apply them to both the OOF predictions (to print our local QWK score) and the test predictions.

Finally, we patch a known issue where Excel/CSV parsing destroyed specific IDs (e.g., turning them into scientific notation) and save the submission.

In [ ]:
print("\nOptimizing Thresholds for QWK...")
optR = OptimizedRounder()
optR.fit(oof_preds, train['target'].values)
coefficients = optR.coef_.x
print(f"Optimal Decision Thresholds found: {coefficients}")

oof_preds_rounded = optR.predict(oof_preds, coefficients)
local_qwk = cohen_kappa_score(train['target'], oof_preds_rounded, weights='quadratic')
print(f"\n🏆 Final Out-Of-Fold QWK Score: {local_qwk:.5f}")

test_preds_rounded = optR.predict(test_preds, coefficients)

submission = pd.DataFrame({
    'A': test['A'].astype(str),
    'target': test_preds_rounded
})

# FIX: Manually patch the Excel-corrupted IDs so Kaggle evaluator accepts them
id_corrections = {
    '4370442': '04370442',
    '6.42E+11': '641697e6'
}
submission['A'] = submission['A'].replace(id_corrections)

output_path = '/kaggle/working/submission.csv'
submission.to_csv(output_path, index=False)
print(f"\nSuccess! Predictions saved to '{output_path}'. You can now submit this file.")